# AkidaNet on ImageNet - Hardware Benchmark

<p align="right">Run Time: ~10 minutes</p>

Measures what an AkidaNet ImageNet model actually costs on Akida hardware: inference
latency, the mapping onto Neural Processors, power, and where the time goes layer by layer.

This notebook needs no dataset setup - it runs from a 10-image sample pack fetched
automatically.

> **Requires a connected AKD1500 device.** The power measurements are read live over I2C,
> so this will not run on Colab. Everything up to and including the sparsity analysis works
> on the software backend; the benchmark sections are skipped without hardware.

In [ ]:
# Colab-only setup. Local users: ignore this cell - it does nothing for you.
import sys
if 'google.colab' in sys.modules:
    !wget -q https://raw.githubusercontent.com/Brainchip-Inc/brainchip_devhub/main/akida1/model_zoo/imagenet_akidanet/colab_setup.py
import colab_setup
colab_setup.setup()

## Configuration

In [ ]:
import numpy as np
import akida

ALPHA = 1.0          # 0.25, 0.5 or 1.0
RESOLUTION = 224     # 160 or 224

NUM_SAMPLES = 100    # inferences per benchmark
POWER_REPEATS = 10
CLOCK_FREQUENCY = 400e6   # 400 MHz for AKD1500

INPUT_SHAPE = (RESOLUTION, RESOLUTION, 3)
print(f'Model: alpha={ALPHA}, {RESOLUTION}x{RESOLUTION}')

## Model

The converted Akida model, straight from `pretrained_models/`. These files are tracked with
Git LFS - if you get a parse error here, run `git lfs pull`.

In [ ]:
from imagenet_akidanet_model import model_path

akida_model = akida.Model(str(model_path(ALPHA, RESOLUTION, 'akida')))
akida_model.summary()

## Samples

Akida is event-driven: a zero activation produces no event and therefore no work. Latency
and power both depend on how sparse the activations are, which depends on the input. So the
benchmark must run on real images - random noise gives the wrong activity statistics and
therefore the wrong numbers.

Ten real images, cycled, capture those statistics well enough. Pass `data_path` to
`get_samples` to draw from the ImageNet validation set instead, if you have it set up.

In [ ]:
from imagenet_akidanet_data import get_samples

samples = get_samples(INPUT_SHAPE, num_samples=NUM_SAMPLES)
print(f'{samples.shape} {samples.dtype}, range [{samples.min()}, {samples.max()}]')

## Hardware device detection

`get_akida_device` checks for a device matching the model's IP version, so a v1 model does
not get mapped onto v2 silicon.

In [ ]:
from brainchip_utils.hardware_utils import get_akida_device

device = get_akida_device(target_version=akida_model.ip_version)
if device is not None:
    print(f'Akida hardware found: {device}')
else:
    print('No hardware found - the benchmark sections below will be skipped.')

## Activation sparsity

Worth looking at before the timings, because it predicts them. Layers with high sparsity
are the ones the hardware gets through quickly.

In [ ]:
from akida_models.sparsity import compute_sparsity
from brainchip_utils.plot_utils import pretty_print_sparsity

sparsity_dict = compute_sparsity(akida_model, samples=samples)
pretty_print_sparsity(sparsity_dict)
print(f'\nMean activation sparsity: {np.mean(list(sparsity_dict.values())) * 100:.2f}%')

## Simple benchmark

A first timing pass in `Minimal` mapping mode at batch size 1, comparing two clocks: the
host's `perf_counter_ns` and the device's own inference clock counter. They should agree
closely - if they do not, the gap is host-side overhead rather than device time.

In [ ]:
import time

if device is not None:
    akida_model.map(device, mode=akida.MapMode.Minimal, hw_only=True)

    inf_clks, inf_times = [], []
    for rr in range(NUM_SAMPLES):
        start_t = time.perf_counter_ns()
        akida_model.forward(samples[rr:rr + 1], batch_size=1)
        inf_times.append(time.perf_counter_ns() - start_t)
        inf_clks.append(akida_model.metrics['inference_clk'])

    mean_inf_clk = np.mean(inf_clks) / CLOCK_FREQUENCY * 1e3   # s to ms
    mean_inf_time = np.mean(inf_times) * 1e-6                  # ns to ms
    print(f'Mean inference time (system clock):        {mean_inf_time:.3f} ms')
    print(f'Mean on-chip time (via chip clock cycles): {mean_inf_clk:.3f} ms')
else:
    print('No hardware - skipping.')

## Full model benchmark

Three mapping modes, measured with power:

- **Minimal** packs the model onto the fewest Neural Processors it fits in, keeping power low.
- **AllNPs** spreads it across more NPs while keeping the number of hardware passes down,
  raising power somewhat but cutting latency roughly in proportion.
- **HwPr** also uses every NP it can, but splits the work over more passes rather than
  minimising them, which can cut latency further still.

None is universally better - it is a latency/power trade-off you choose per application.

In [ ]:
from brainchip_utils.hardware_utils import full_model_benchmark, get_mapping_stats

full_results = {}
if device is not None:
    for mm in ['Minimal', 'AllNps', 'HwPr']:
        map_mode = getattr(akida.MapMode, mm)
        print(f'Running full-model benchmark (MapMode={mm}, {POWER_REPEATS} repeats)...')
        res = full_model_benchmark(akida_model, device, samples,
                                   map_mode=map_mode, repeats=POWER_REPEATS)
        if res is None:
            # Not every mode maps every model onto a single hardware sequence
            print(f'  MapMode={mm} did not map to hardware - skipping this mode.')
            continue
        full_results[mm] = res
        # Re-map without hw_only so akida_model.sequences is populated for the stats
        akida_model.map(device, mode=map_mode)
        num_nps, num_passes, num_sequences = get_mapping_stats(akida_model)
        full_results[mm]['num_nps'] = num_nps
        full_results[mm]['num_passes'] = num_passes
        print(f'  {num_nps} NP(s), {num_passes} pass(es), {num_sequences} sequence(s)')
        if num_sequences > 1:
            print('  WARNING: model not completely mapped to hardware')
else:
    print('No hardware - skipping.')

In [ ]:
from brainchip_utils.plot_utils import plot_full_model_results

if device is not None:
    plot_full_model_results(full_results, akida_model, device,
                            model_name=f'akidanet_imagenet alpha={ALPHA} {RESOLUTION}px')

## Per-layer benchmark

Where the time actually goes. Read this alongside the sparsity numbers above: the
expensive layers are usually the dense early ones, before activations have become sparse.

In [ ]:
from brainchip_utils.hardware_utils import per_layer_benchmark

if device is not None:
    akida_model.map(device, mode=akida.MapMode.Minimal, hw_only=True)
    print(f'Running per-layer benchmark ({NUM_SAMPLES} samples)...')
    per_layer_results = per_layer_benchmark(akida_model, device, samples, repeats=NUM_SAMPLES)
else:
    print('No hardware - skipping.')

In [ ]:
from brainchip_utils.plot_utils import plot_per_layer_results

if device is not None:
    # Map without hw_only so the mapping plot can be drawn
    akida_model.map(device, mode=akida.MapMode.Minimal)
    plot_per_layer_results(per_layer_results, akida_model, sparsity_dict,
                           model_name=f'akidanet_imagenet alpha={ALPHA} {RESOLUTION}px')

## Summary

In [ ]:
print(f'AkidaNet alpha={ALPHA}, {RESOLUTION}x{RESOLUTION}')
print(f'  mean activation sparsity: {np.mean(list(sparsity_dict.values())) * 100:.2f}%')
if device is not None:
    for mm, res in full_results.items():
        line = (f'  {mm:8} {res["num_nps"]:3d} NPs, '
                f'{res["mean_clk_ms"]:7.3f} ms/inference')
        if res['power'] is not None:
            line += (f', {res["power"]["avg_total_mw"]:6.1f} mW total'
                     f', {res["power"]["avg_energy_mj"]:6.3f} mJ/inference')
        print(line)
else:
    print('  no hardware present - latency and power not measured')

To record these numbers into the README Model Card, run the benchmark script with
`--save-metrics` and regenerate the README:

```bash
python imagenet_akidanet_benchmark.py -a 1.0 -i 224 --save-metrics
python update_readme.py
```